In [ ]:
"""
====================================================================================
Compute Attacker Profit in USD from Token Profits using Moralis API
====================================================================================

Purpose
-------
This module calculates the USD equivalent of attacker profits in DeFi transactions
by fetching historical token prices from the Moralis EVM API. It allows for
batch-wise computation and can return partial results in case of interruption or
API failure.

Workflow
--------
1. Iterate over each row in the input DataFrame containing detected attacker profits.
2. For each transaction:
   - Extract the profit token, block number, and profit amount.
   - Skip rows with missing or non-positive profits.
   - Fetch the token price in USD at the specified block using Moralis API.
   - Compute USD profit as profit_amount * token_price.
3. Return a new DataFrame with an additional column `attacker_profit_usd`.
4. Partial results are returned if an API error occurs mid-processing.

Inputs
------
- `df`: DataFrame with at least the columns:
    - `profit_token`: Token address of the profit token.
    - `block`: Block number at which the profit was realized.
    - `attacker_profit`: Profit amount in token units.
- `api_key`: Moralis API key for authentication.
- `chain` (optional): Blockchain identifier (default `"eth"`).
- `batch_size` (optional): Interval for intermediate progress reporting.

Outputs
-------
- DataFrame with an added column `attacker_profit_usd` containing the computed
  profit in USD for each row.

Use Case
--------
- Enables quantification of attacker profits in USD for DeFi and MEV research.
- Supports batch processing for large datasets with optional progress feedback.
- Returns partially filled results on failure for safe incremental computation.
"""


In [ ]:
from moralis import evm_api
import pandas as pd
from tqdm import tqdm

def compute_attacker_profit_usd(df: pd.DataFrame, api_key: str, chain="eth", batch_size=10):
    usd_profits = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Computing USD profit"):
        profit_token = row.get('profit_token')
        block = row.get('block')
        profit_amount = row.get('attacker_profit')

        if profit_token is None or profit_amount is None or profit_amount <= 0:
            usd_profits.append(None)
            continue

        try:
            params = {
                "chain": chain,
                "include": "percent_change",
                "to_block": block,
                "address": profit_token
            }

            result = evm_api.token.get_token_price(api_key=api_key, params=params)
            usd_price = result.get("usdPrice", None)
            usd_profit = profit_amount * usd_price if usd_price is not None else None
            usd_profits.append(usd_profit)

        except Exception:
            df_partial = df.copy()
            df_partial['attacker_profit_usd'] = usd_profits + [None] * (len(df) - len(usd_profits))
            return df_partial

        if len(usd_profits) % batch_size == 0:
            pass  # optional progress feedback

    df_final = df.copy()
    df_final['attacker_profit_usd'] = usd_profits
    return df_final
